## Qualidade de Dados — Validação

Cinco validações realizadas nas camadas Silver e Gold, garantindo qualidade e consistência dos dados.

**1ª Validação:** Integridade dos Dados

Verificação dos registros descartados entre Bronze e Silver por duplicidade ou falta de chave primária.

**Resultado ideal:** mesmo número de linhas ou menos (nunca mais).

In [0]:
%sql
SELECT 'orders' AS tabela,
       (SELECT COUNT(*) FROM delivery_center.bronze.orders) AS linhas_bronze,
       (SELECT COUNT(*) FROM delivery_center.silver.orders) AS linhas_silver
UNION ALL
SELECT 'deliveries',
       (SELECT COUNT(*) FROM delivery_center.bronze.deliveries),
       (SELECT COUNT(*) FROM delivery_center.silver.deliveries)
UNION ALL
SELECT 'stores',
       (SELECT COUNT(*) FROM delivery_center.bronze.stores),
       (SELECT COUNT(*) FROM delivery_center.silver.stores)
UNION ALL
SELECT 'hubs',
       (SELECT COUNT(*) FROM delivery_center.bronze.hubs),
       (SELECT COUNT(*) FROM delivery_center.silver.hubs)
UNION ALL
SELECT 'drivers',
       (SELECT COUNT(*) FROM delivery_center.bronze.drivers),
       (SELECT COUNT(*) FROM delivery_center.silver.drivers)
UNION ALL
SELECT 'channels',
       (SELECT COUNT(*) FROM delivery_center.bronze.channels),
       (SELECT COUNT(*) FROM delivery_center.silver.channels)
UNION ALL
SELECT 'payments',
       (SELECT COUNT(*) FROM delivery_center.bronze.payments),
       (SELECT COUNT(*) FROM delivery_center.silver.payments);

tabela,linhas_bronze,linhas_silver
orders,368999,368999
deliveries,378843,358654
stores,951,951
hubs,32,32
drivers,4824,4824
channels,40,40
payments,400834,400834


**2ª Validação:** Conversão de Datas e Horários

Contagem de valores nulos gerados na conversão dos 8 campos order_moment_* para TIMESTAMP, comparando com a quantidade de nulos já existente na tabela Bronze (antes da conversão).

**Resultado ideal:** diferença igual a zero entre os nulos do Bronze e os nulos do Silver

In [0]:
%sql
WITH bronze_nulos AS (
    SELECT
        COUNT(*) AS total_linhas,
        SUM(CASE WHEN order_moment_created IS NULL OR TRIM(order_moment_created) = '' THEN 1 ELSE 0 END)       AS order_moment_created,
        SUM(CASE WHEN order_moment_accepted IS NULL OR TRIM(order_moment_accepted) = '' THEN 1 ELSE 0 END)      AS order_moment_accepted,
        SUM(CASE WHEN order_moment_ready IS NULL OR TRIM(order_moment_ready) = '' THEN 1 ELSE 0 END)            AS order_moment_ready,
        SUM(CASE WHEN order_moment_collected IS NULL OR TRIM(order_moment_collected) = '' THEN 1 ELSE 0 END)    AS order_moment_collected,
        SUM(CASE WHEN order_moment_in_expedition IS NULL OR TRIM(order_moment_in_expedition) = '' THEN 1 ELSE 0 END) AS order_moment_in_expedition,
        SUM(CASE WHEN order_moment_delivering IS NULL OR TRIM(order_moment_delivering) = '' THEN 1 ELSE 0 END)  AS order_moment_delivering,
        SUM(CASE WHEN order_moment_delivered IS NULL OR TRIM(order_moment_delivered) = '' THEN 1 ELSE 0 END)    AS order_moment_delivered,
        SUM(CASE WHEN order_moment_finished IS NULL OR TRIM(order_moment_finished) = '' THEN 1 ELSE 0 END)      AS order_moment_finished
    FROM delivery_center.bronze.orders
),
silver_nulos AS (
    SELECT
        SUM(CASE WHEN order_moment_created IS NULL THEN 1 ELSE 0 END)       AS order_moment_created,
        SUM(CASE WHEN order_moment_accepted IS NULL THEN 1 ELSE 0 END)      AS order_moment_accepted,
        SUM(CASE WHEN order_moment_ready IS NULL THEN 1 ELSE 0 END)         AS order_moment_ready,
        SUM(CASE WHEN order_moment_collected IS NULL THEN 1 ELSE 0 END)     AS order_moment_collected,
        SUM(CASE WHEN order_moment_in_expedition IS NULL THEN 1 ELSE 0 END) AS order_moment_in_expedition,
        SUM(CASE WHEN order_moment_delivering IS NULL THEN 1 ELSE 0 END)    AS order_moment_delivering,
        SUM(CASE WHEN order_moment_delivered IS NULL THEN 1 ELSE 0 END)     AS order_moment_delivered,
        SUM(CASE WHEN order_moment_finished IS NULL THEN 1 ELSE 0 END)      AS order_moment_finished
    FROM delivery_center.silver.orders
),
comparativo AS (
    SELECT
        b.total_linhas,
        b.order_moment_created       AS created_bronze,       s.order_moment_created       AS created_silver,
        b.order_moment_accepted      AS accepted_bronze,      s.order_moment_accepted      AS accepted_silver,
        b.order_moment_ready         AS ready_bronze,         s.order_moment_ready         AS ready_silver,
        b.order_moment_collected     AS collected_bronze,     s.order_moment_collected     AS collected_silver,
        b.order_moment_in_expedition AS in_expedition_bronze, s.order_moment_in_expedition AS in_expedition_silver,
        b.order_moment_delivering    AS delivering_bronze,    s.order_moment_delivering    AS delivering_silver,
        b.order_moment_delivered     AS delivered_bronze,     s.order_moment_delivered     AS delivered_silver,
        b.order_moment_finished      AS finished_bronze,      s.order_moment_finished      AS finished_silver
    FROM bronze_nulos b
    CROSS JOIN silver_nulos s
)
SELECT
    coluna,
    nulos_bronze,
    nulos_silver,
    (nulos_silver - nulos_bronze) AS diferenca,
    ROUND((nulos_silver - nulos_bronze) / total_linhas * 100, 2) AS percentual_diferenca_sobre_total
FROM comparativo
LATERAL VIEW STACK(
    8,
    'order_moment_created', created_bronze, created_silver,
    'order_moment_accepted', accepted_bronze, accepted_silver,
    'order_moment_ready', ready_bronze, ready_silver,
    'order_moment_collected', collected_bronze, collected_silver,
    'order_moment_in_expedition', in_expedition_bronze, in_expedition_silver,
    'order_moment_delivering', delivering_bronze, delivering_silver,
    'order_moment_delivered', delivered_bronze, delivered_silver,
    'order_moment_finished', finished_bronze, finished_silver
) s AS coluna, nulos_bronze, nulos_silver
ORDER BY diferenca DESC;

coluna,nulos_bronze,nulos_silver,diferenca,percentual_diferenca_sobre_total
order_moment_delivered,349398,349398,0,0.0
order_moment_delivering,25316,25316,0,0.0
order_moment_collected,42894,42894,0,0.0
order_moment_ready,25106,25106,0,0.0
order_moment_accepted,9461,9461,0,0.0
order_moment_in_expedition,67429,67429,0,0.0
order_moment_created,0,0,0,0.0
order_moment_finished,15599,15599,0,0.0


**3ª Validação:** Consistência das Chaves Estrangeiras

Pedidos em fact_orders sem correspondência válida em alguma dimensão (store_id, driver_id, channel_id ou date_id nulos depois do join).

**Resultado ideal:** zero pedidos sem correspondência em cada dimensão.

In [0]:
%sql
SELECT
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END)   AS sem_store_id,
    SUM(CASE WHEN driver_id IS NULL THEN 1 ELSE 0 END)  AS sem_driver_id,
    SUM(CASE WHEN channel_id IS NULL THEN 1 ELSE 0 END) AS sem_channel_id,
    SUM(CASE WHEN date_id IS NULL THEN 1 ELSE 0 END)    AS sem_date_id
FROM delivery_center.gold.fact_orders;

total_pedidos,sem_store_id,sem_driver_id,sem_channel_id,sem_date_id
368999,0,25583,0,0


**4ª Validação:** Valores numéricos suspeitos

Contagem de valores negativos em campos que não fazem sentido de negócio (valor do pedido, taxa de entrega, custo de entrega, distância da entrega).

**Resultado ideal:** zero valores negativos em todos os campos.

In [0]:
%sql
SELECT
    SUM(CASE WHEN order_amount < 0 THEN 1 ELSE 0 END)             AS valor_pedido_negativo,
    SUM(CASE WHEN order_delivery_fee < 0 THEN 1 ELSE 0 END)       AS taxa_entrega_negativa,
    SUM(CASE WHEN order_delivery_cost < 0 THEN 1 ELSE 0 END)      AS custo_entrega_negativo,
    SUM(CASE WHEN delivery_distance_meters < 0 THEN 1 ELSE 0 END) AS distancia_negativa
FROM delivery_center.gold.fact_orders;

valor_pedido_negativo,taxa_entrega_negativa,custo_entrega_negativo,distancia_negativa
0,0,0,0


**5ª Validação:** Unicidade das dimensões

Confirmação de que nenhuma dimensão do modelo Estrela tem chave primária duplicada.

**Resultado ideal:** total de linhas igual ao total de valores distintos da chave primária, em cada dimensão (`dim_stores`, `dim_drivers`, `dim_channels`, `dim_date`).

In [0]:
%sql
SELECT 'dim_stores' AS tabela, COUNT(*) AS total, COUNT(DISTINCT store_id) AS distintos FROM delivery_center.gold.dim_stores
UNION ALL SELECT 'dim_drivers', COUNT(*), COUNT(DISTINCT driver_id) FROM delivery_center.gold.dim_drivers
UNION ALL SELECT 'dim_channels', COUNT(*), COUNT(DISTINCT channel_id) FROM delivery_center.gold.dim_channels
UNION ALL SELECT 'dim_date', COUNT(*), COUNT(DISTINCT date_id) FROM delivery_center.gold.dim_date;

tabela,total,distintos
dim_stores,951,951
dim_drivers,4824,4824
dim_channels,40,40
dim_date,120,120
